# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 4 — Embeddings e ingesta documental

ITBA Educación Ejecutiva

Preparación de información para recuperación semántica: hoy dejamos de hablarle al modelo de a una consulta por vez, y empezamos a construir el corpus sobre el que va a buscar.

### Cómo se usa este notebook

- El docente corre y explica cada celda en vivo (demo con AWS Bedrock). No programás en simultáneo durante la clase.
- Celdas **🏠 Practicá**: tarea individual para después de la clase.
- **Sección 12 (Laboratorio)**: tarea principal de la semana — checkpoint del proyecto integrador. Se resuelve con el proveedor de cada uno.
- **Marco teórico**: [`apunte.md`](apunte.md) (mismas secciones 1–11 que este notebook), con las fuentes anotadas al principio.

## Objetivo de la clase

Al final de la clase vas a poder:
- Explicar qué es un embedding y por qué la similitud coseno mide "cercanía de significado".
- Limpiar y normalizar texto antes de vectorizarlo.
- Elegir una estrategia de chunking (tamaño, solapamiento) con criterio, no por default.
- Diseñar metadata que permita trazar cada fragmento hasta su documento de origen.
- Separar claramente el pipeline de ingesta del de consulta.

**Checkpoint del proyecto integrador de hoy**: pipeline de ingesta del corpus (limpieza, chunking y metadata) — ver `proyecto-integrador/README.md`, checkpoint Clase 4.

## Agenda

1. Información estructurada vs. no estructurada
2. Embeddings y similitud semántica
3. Tu proveedor: embeddings — referencia rápida
4. Dimensión de los embeddings: Matryoshka
5. Limpieza y normalización
6. Estrategias de chunking y solapamiento
7. Metadata, identificadores y trazabilidad
8. Separación entre ingesta y consulta
9. Comparación de estrategias de chunking
10. Chunking por tokens, no por caracteres
11. Chunking semántico — implementado
12. Laboratorio: pipeline de ingesta documental

## 1. Información estructurada vs. no estructurada

- **Estructurada**: tablas, JSON, filas de una base de datos — ya tiene un esquema. Se consulta con SQL/filtros exactos, no necesita embeddings.
- **No estructurada**: texto libre (documentación, tickets, emails, PDFs) — no tiene un esquema uniforme. Es el caso que nos ocupa de acá en adelante: para "buscar por significado" en texto libre necesitamos convertirlo a una representación que capture semántica, no solo keywords.

El corpus del TP integrador (documentación técnica) es no estructurado — eso es justamente lo que hace necesario todo lo que viene en esta clase.

## 2. Embeddings y similitud semántica

Un embedding es un vector $\vec{v} \in \mathbb{R}^d$ que representa el significado de un texto. Textos semánticamente parecidos quedan "cerca" en ese espacio — se mide con **similitud coseno**:

$$
\text{sim}(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{\lVert \vec{a} \rVert \, \lVert \vec{b} \rVert}
$$

El coseno va de −1 a 1: cercano a 1 → misma dirección → significado parecido. En teoría, cercano a 0 → sin relación; **en la práctica**, los modelos de embeddings son *anisotrópicos* (sus vectores ocupan un cono estrecho del espacio) y textos sin relación dan valores claramente positivos. Lo que se interpreta es el **orden relativo**, no el número (apunte §2.4). (Esto ya se mencionó en la Clase 1 — hoy lo usamos en código real por primera vez.)

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()

import boto3
bedrock = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
EMBED_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_EMBED", "cohere.embed-v4:0")

def embed_bedrock(textos: list[str], input_type: str = "search_document") -> list[list[float]]:
    """Devuelve un embedding (lista de floats) por cada texto de la lista, usando Cohere Embed v4 en Bedrock."""
    body = json.dumps({"texts": textos, "input_type": input_type, "embedding_types": ["float"]})
    resp = bedrock.invoke_model(modelId=EMBED_MODEL_ID, body=body, accept="*/*", contentType="application/json")
    resp_body = json.loads(resp["body"].read())
    return resp_body["embeddings"]["float"]

In [ ]:
import numpy as np

def similitud_coseno(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

frases = [
    "La ventana de contexto limita cuánto texto puede procesar un modelo por llamada.",
    "El límite de tokens de entrada y salida define cuánta información cabe en una consulta.",
    "El dulce de leche se prepara cocinando leche con azúcar a fuego lento durante horas.",
]

vectores = embed_bedrock(frases)
print(f"Dimensión de cada embedding: {len(vectores[0])}")

print("\nSimilitud entre frase 1 y 2 (ambas sobre contexto/tokens):", round(similitud_coseno(vectores[0], vectores[1]), 3))
print("Similitud entre frase 1 y 3 (temas no relacionados):       ", round(similitud_coseno(vectores[0], vectores[2]), 3))

### ¿De dónde sale esa geometría?

Un modelo de embeddings no "sabe" de antemano qué textos son parecidos — lo aprende con **aprendizaje contrastivo**: durante el entrenamiento se le muestran pares de textos que *deberían* quedar cerca (ej. una pregunta y su respuesta correcta, o dos traducciones del mismo texto) y pares que *no* deberían (**negativos**). En la práctica, los negativos salen de dos lados: los positivos de los *otros* ejemplos del mismo lote (*in-batch negatives*, gratis y abundantes) y **negativos difíciles** (*hard negatives*: textos parecidos en la superficie pero no relevantes, que enseñan las distinciones finas). La función de pérdida más usada, **InfoNCE**, es literalmente una clasificación: "entre todos estos candidatos, ¿cuál es el positivo?" (fórmula en el apunte §2.3). El objetivo de entrenamiento empuja los pares positivos a acercarse en el espacio vectorial y los negativos a alejarse — después de millones de estos ejemplos, la geometría resultante captura similitud semántica como efecto secundario de optimizar ese objetivo, no porque alguien le haya dicho explícitamente "closures y scope están relacionados".

Esto explica por qué los embeddings de dos proveedores distintos **no son comparables**: cada uno aprendió su propia geometría, con sus propios pares de entrenamiento — no hay garantía de que compartan ni siquiera la noción de "cerca".

**🏠 Practicá**: agregá una cuarta frase que hable de otra cosa totalmente distinta (ej. una receta de cocina) y confirmá que su similitud con las otras tres es la más baja del grupo.

## 3. Tu proveedor: embeddings — referencia rápida

Igual que con generación (Clase 2), elegí el proveedor que tengas disponible. **Anthropic no tiene modelo de embeddings propio** — si tu proveedor de generación es Anthropic, combinalo con cualquiera de los otros (Cohere en Bedrock, OpenAI, Google u Ollama) para esta parte.

**Ojo con la asimetría consulta–documento**: muchos modelos esperan que les digas si el texto es una *consulta* o un *documento* — Cohere con `input_type`, Gemini con `task_type`, `nomic-embed-text` con prefijos en el texto. Omitirlo **no da error**: solo empeora la búsqueda en silencio (apunte §3).

```bash
pip install numpy       # ya instalado si veniás de clases anteriores
pip install openai      # si usás OpenAI
pip install google-genai
pip install ollama
```

In [ ]:
# Cada proveedor en un bloque try/except: si no tenés instalado su SDK, la celda no falla.
# Todas las funciones reciben input_type ("search_document" o "search_query") para respetar la asimetría.

# --- OpenAI (sus modelos de embeddings son simétricos: no distinguen consulta de documento) ---
try:
    from openai import OpenAI

    def embed_openai(textos: list[str], input_type: str = "search_document") -> list[list[float]]:
        client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        resp = client.embeddings.create(model=os.environ.get("OPENAI_EMBEDDING_MODEL_ID", "text-embedding-3-large"), input=textos)
        return [d.embedding for d in resp.data]
except ImportError:
    print("openai no instalado (pip install openai) — se saltea")


# --- Google Gemini: task_type distingue consulta de documento ---
try:
    from google import genai
    from google.genai import types

    def embed_google(textos: list[str], input_type: str = "search_document") -> list[list[float]]:
        client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
        tarea = "RETRIEVAL_QUERY" if input_type == "search_query" else "RETRIEVAL_DOCUMENT"
        resp = client.models.embed_content(
            model=os.environ.get("GOOGLE_EMBEDDING_MODEL_ID", "gemini-embedding-001"),
            contents=textos,
            config=types.EmbedContentConfig(task_type=tarea),
        )
        return [e.values for e in resp.embeddings]
except ImportError:
    print("google-genai no instalado (pip install google-genai) — se saltea")


# --- Ollama (local): nomic-embed-text espera PREFIJOS en el texto ---
try:
    import ollama

    def embed_ollama(textos: list[str], input_type: str = "search_document") -> list[list[float]]:
        prefijo = "search_query: " if input_type == "search_query" else "search_document: "
        resp = ollama.embed(model=os.environ.get("OLLAMA_EMBEDDING_MODEL_ID", "nomic-embed-text"),
                            input=[prefijo + t for t in textos])
        return resp.embeddings
except ImportError:
    print("ollama no instalado (pip install ollama) — se saltea")

**🏠 Practicá**: corré la misma lista de `frases` de la sección 2 con la función de tu proveedor y recalculá las similitudes coseno. Los valores exactos van a diferir de los de Bedrock (cada modelo de embeddings vive en su propio espacio vectorial — nunca mezcles embeddings de dos proveedores distintos en el mismo índice), pero el **orden relativo** (frase 1 más parecida a frase 2 que a frase 3) debería mantenerse.

## 4. Dimensión de los embeddings: Matryoshka

Cohere Embed 4 (Clase 1) permite elegir la dimensión del vector de salida: 256, 512, 1024 o 1536. No es un parámetro de compresión posterior — el modelo está entrenado específicamente para que esto funcione bien, con una técnica llamada **Matryoshka Representation Learning (MRL)**.

La idea: en vez de entrenar un embedding de 1536 dimensiones donde la información relevante está repartida sin orden entre todas ellas, MRL entrena el vector para que **los primeros 256 valores ya sean, por sí solos, un embedding razonable** — y agregar los siguientes 256 lo va refinando, como las muñecas rusas que le dan el nombre. Esto significa que truncar el vector (quedarte con las primeras $d$ dimensiones) da una versión más chica y más rápida de buscar, con una degradación de calidad gradual y controlada — no un embedding roto. (Si truncás el vector **a mano** en vez de pedir `output_dimension`, deja de tener norma 1: con coseno no importa, pero con producto interno o distancia L2 hay que **re-normalizarlo** — apunte §4.)

**Por qué importa en la práctica**: dimensión más chica = menos espacio de almacenamiento y búsquedas más rápidas (la distancia coseno se calcula sobre menos números) a costa de algo de precisión. Para un corpus chico (como el del TP) la diferencia puede ser insignificante; para un corpus de millones de chunks, pasar de 1536 a 256 dimensiones puede ser la diferencia entre un índice que entra en memoria y uno que no.

In [ ]:
def embed_bedrock_con_dimension(textos: list[str], output_dimension: int, input_type: str = "search_document") -> list[list[float]]:
    body = json.dumps({
        "texts": textos,
        "input_type": input_type,
        "embedding_types": ["float"],
        "output_dimension": output_dimension,
    })
    resp = bedrock.invoke_model(modelId=EMBED_MODEL_ID, body=body, accept="*/*", contentType="application/json")
    return json.loads(resp["body"].read())["embeddings"]["float"]


frases_prueba = frases  # las 3 frases de la sección 2
consulta_prueba = "¿cómo se limita la cantidad de texto que procesa un modelo por llamada?"

for dim in [256, 1024, 1536]:
    vectores_dim = embed_bedrock_con_dimension(frases_prueba, output_dimension=dim)
    vec_consulta = embed_bedrock_con_dimension([consulta_prueba], output_dimension=dim, input_type="search_query")[0]
    similitudes = [similitud_coseno(vec_consulta, v) for v in vectores_dim]
    mejor = similitudes.index(max(similitudes))
    print(f"dim={dim:>4}  mejor match: frase {mejor}  (sim={similitudes[mejor]:.3f})  -- todas: {[round(s,3) for s in similitudes]}")

Con un ejemplo tan chico, es esperable que las tres dimensiones elijan la misma frase como más relevante — la pregunta real para un corpus grande es si el **ranking completo** (no solo el top-1) se mantiene estable al bajar la dimensión. Esa es, justamente, la practicá de hoy.

**🏠 Practicá**: embeb el `documento_largo` de la sección 9 con `output_dimension=256` y con `output_dimension=1536`, y compará el top-3 recuperado para la misma consulta en ambos casos — ¿cambia el orden? ¿cambian los chunks devueltos?

## 5. Limpieza y normalización

Antes de vectorizar, conviene limpiar el texto — no es cosmético, afecta directamente la calidad de la búsqueda:

- **Espacios y saltos de línea redundantes**: texto extraído de PDF/HTML suele traer basura de formato.
- **Boilerplate repetido** (headers/footers, "Página X de Y", menús de navegación): si aparece en todos los chunks, contamina la similitud — todo se parece un poco entre sí por el ruido compartido.
- **Normalización Unicode**: el mismo texto puede estar codificado de varias formas ("é" como un carácter, o como "e" + acento combinante; la ligadura "ﬁ" en vez de "fi"). `unicodedata.normalize` las unifica. **No** convierte comillas tipográficas (“ ”) en rectas ni repara texto mal decodificado como "canciÃ³n" (para eso existe la librería `ftfy`). Y la forma NFKC puede **cambiar el significado**: convierte "x²" en "x2" (apunte §5.2).
- **¿Lowercase?** Depende: para texto natural generalmente no hace falta (los embeddings modernos son robustos a mayúsculas), pero para código o identificadores técnicos, bajar a minúsculas puede destruir información relevante (`API_KEY` ≠ `api_key` en un fragmento de código). No lo hagas por default sin pensarlo.

In [ ]:
import re
import unicodedata

def limpiar_texto(texto: str, forma_unicode: str = "NFC") -> str:
    texto = unicodedata.normalize(forma_unicode, texto)   # NFC: unifica acentos; NFKC además ligaduras, ancho completo, etc.
    texto = re.sub(r"[ \t]+", " ", texto)                 # espacios repetidos
    texto = re.sub(r"[ \t]+\n", "\n", texto)               # espacios al final de línea (si no, \n \n \n no colapsa)
    texto = re.sub(r"\n{3,}", "\n\n", texto)               # más de 2 saltos de línea seguidos
    return texto.strip()


texto_sucio = """   Página 1 de 3


Guía  de   configuración


Para configurar  las credenciales,   copiá .env.example a .env.\n\n\n\nCompletá la sección de tu proveedor.
"""

print(repr(limpiar_texto(texto_sucio)))

# Qué hace (y qué no) la normalización Unicode
ejemplo = "e\u0301 | ﬁnal | Ａ | x² | “comillas” | canciÃ³n"
print("\noriginal:", ejemplo)
print("NFC     :", unicodedata.normalize("NFC", ejemplo), " <- solo unifica el acento combinante")
print("NFKC    :", unicodedata.normalize("NFKC", ejemplo), " <- ligadura, ancho completo y x² cambian; comillas y mojibake NO")

**🏠 Practicá**: agregá una regla a `limpiar_texto` que detecte y elimine una línea de boilerplate recurrente (ej. `"Página X de Y"` con una regex) — es el tipo de ruido que vas a encontrar en PDFs reales del corpus del TP.

## 6. Estrategias de chunking y solapamiento

Un documento entero casi nunca entra (ni conviene que entre) en un solo embedding — hay que partirlo en **chunks**. Dos decisiones centrales:

- **Tamaño del chunk**: muy chico → pierde contexto (una oración sola puede ser ambigua). Muy grande → mezcla varios temas en un solo vector, diluyendo la similitud con cualquiera de ellos.
- **Solapamiento (overlap)**: sin overlap, una idea que cruza el límite entre dos chunks queda cortada en ambos. Con un poco de overlap (ej. 10-20% del tamaño del chunk), se mitiga ese corte a costa de un poco de redundancia.

In [ ]:
def chunking_por_caracteres(texto: str, tamano: int = 300, solapamiento: int = 50) -> list[str]:
    """Chunking de tamaño fijo con solapamiento, por caracteres (simple, no respeta oraciones)."""
    if not 0 <= solapamiento < tamano:
        raise ValueError("el solapamiento tiene que ser menor que el tamaño")
    chunks = []
    paso = tamano - solapamiento
    for inicio in range(0, len(texto), paso):
        chunk = texto[inicio:inicio + tamano]
        if chunk.strip():
            chunks.append(chunk)
        if inicio + tamano >= len(texto):
            break
    return chunks


def chunking_por_parrafos(texto: str, max_chars: int = 500) -> list[str]:
    """Agrupa párrafos (separados por línea en blanco) hasta llegar al tamaño máximo.
    Un párrafo que solo ya supera max_chars se parte por caracteres."""
    separador = "\n\n"
    parrafos = [p.strip() for p in texto.split(separador) if p.strip()]
    chunks, actual = [], ""
    for p in parrafos:
        piezas = chunking_por_caracteres(p, max_chars, max_chars // 10) if len(p) > max_chars else [p]
        for pieza in piezas:
            if actual and len(actual) + len(separador) + len(pieza) <= max_chars:   # el separador también cuenta
                actual = actual + separador + pieza
            else:
                if actual:
                    chunks.append(actual)
                actual = pieza
    if actual:
        chunks.append(actual)
    return chunks

## 7. Metadata, identificadores y trazabilidad

Un chunk suelto, sin contexto de dónde vino, no sirve para citar fuentes (Clase 6) ni para debuggear por qué el sistema recuperó algo irrelevante (Clase 7). Cada chunk necesita viajar con su metadata.

In [ ]:
import hashlib
from dataclasses import dataclass, field

@dataclass
class Chunk:
    texto: str
    documento_id: str
    posicion: int           # orden del chunk dentro del documento
    chunk_id: str = field(init=False)

    def __post_init__(self):
        # id determinístico: mismo documento + misma posición + mismo contenido -> mismo id, siempre.
        # Ojo: como incluye la posición, si se edita el principio del documento cambian los ids de ahí en adelante
        # (ver la tabla de diseños de ids en el apunte §7).
        hash_contenido = hashlib.sha256(self.texto.encode()).hexdigest()[:8]
        self.chunk_id = f"{self.documento_id}-{self.posicion:03d}-{hash_contenido}"


def construir_chunks(texto: str, documento_id: str, tamano: int = 300, solapamiento: int = 50) -> list[Chunk]:
    textos = chunking_por_caracteres(texto, tamano, solapamiento)
    return [Chunk(texto=t, documento_id=documento_id, posicion=i) for i, t in enumerate(textos)]


corpus_ejemplo = """La Converse API de Bedrock es una interfaz unificada para conversación. Funciona igual para
distintos modelos: cambia el modelId, no la forma de llamar. Soporta mensajes de sistema,
parámetros de inferencia como temperatura y top_p, y tool calling con un schema JSON de argumentos.

Para usarla se necesita un cliente de boto3 apuntando al servicio bedrock-runtime, y credenciales
configuradas vía variables de entorno o un perfil de AWS."""

chunks = construir_chunks(limpiar_texto(corpus_ejemplo), documento_id="doc-converse-api")
for c in chunks:
    print(f"[{c.chunk_id}] (pos {c.posicion}) {c.texto[:60]}...")

**🏠 Practicá**: agregá un campo `fuente: str` (ej. una URL o un nombre de archivo) al dataclass `Chunk`, y otro campo `fecha_ingesta` con un timestamp. Esto es exactamente lo que vas a necesitar en la Clase 6 para implementar citas.

## 8. Separación entre ingesta y consulta

Son dos pipelines distintos, que corren en momentos distintos:

- **Ingesta** (lo de hoy): se corre una vez (o cada vez que el corpus cambia) — limpiar, chunkear, embeber, guardar en un índice. Es costoso en tiempo pero infrecuente.
- **Consulta** (Clase 5 en adelante): se corre en cada pregunta del usuario — embeber la consulta y buscar en el índice ya construido. Tiene que ser rápido.

Error común: mezclar la lógica de las dos etapas en una sola función que "ingesta y busca" — hace que no se pueda optimizar cada una por separado (por ejemplo, batchear la ingesta, cachear embeddings de consultas frecuentes).

## 9. Comparación de estrategias de chunking

No hay un tamaño "correcto" — depende del corpus y de qué tipo de pregunta se espera responder. Comparemos en la práctica.

In [ ]:
documento_largo = "\n\n".join([corpus_ejemplo] * 3)  # simulamos un documento más largo repitiendo el de ejemplo

for tamano, overlap in [(150, 0), (150, 30), (400, 0), (400, 80)]:
    chunks_resultantes = chunking_por_caracteres(documento_largo, tamano=tamano, solapamiento=overlap)
    print(f"tamano={tamano:>4} overlap={overlap:>3} -> {len(chunks_resultantes)} chunks")

**🏠 Practicá**: corré `chunking_por_parrafos` sobre el mismo `documento_largo` con dos valores de `max_chars` distintos, y compará la cantidad de chunks contra `chunking_por_caracteres`. ¿En qué casos el chunking por párrafos corta "mejor" (en límites de idea) que el de tamaño fijo?

## 10. Chunking por tokens, no por caracteres

Todo el chunking de la sección 6 cuenta **caracteres**. Pero lo que realmente limita (la ventana de contexto del modelo generador, el máximo de tokens por documento del modelo de embeddings) se mide en **tokens**, no en caracteres — y la relación entre ambos no es fija.

Un chunk de "300 caracteres" puede ser:
- ~75 tokens si es texto en **inglés** simple ("~4 caracteres por token" es una heurística para inglés con los tokenizadores de OpenAI).
- Más tokens si es **castellano**: el mismo texto ocupa más tokens que en inglés (la disparidad entre idiomas puede ser muy grande — apunte §10).
- Bastante más de 75 tokens si tiene código, símbolos, o palabras técnicas poco frecuentes (BPE las fragmenta más — Clase 1).

Contar caracteres es una aproximación cómoda pero puede subestimar o sobreestimar el uso real de tokens — y si el corpus real del TP (MDN, con bloques de código JavaScript mezclados con prosa) tiene ambos tipos de contenido, el error se nota.

In [ ]:
# %pip install tiktoken
import tiktoken

# cl100k_base es el tokenizador de modelos de OpenAI anteriores. Claude, Cohere y GPT-4o (o200k_base)
# tokenizan distinto: los conteos son una APROXIMACIÓN. Dejá margen, o usá el contador de tu proveedor.
_encoder = tiktoken.get_encoding("cl100k_base")

def contar_tokens(texto: str) -> int:
    return len(_encoder.encode(texto))


# Por qué NO cortar en posiciones de token crudas: BPE trabaja sobre BYTES, y una "ñ" ocupa 2 bytes en UTF-8
tokens_nandu = _encoder.encode(" ñandú")            # con el espacio previo, como aparece en medio de una oración
print("' ñandú' ->", [_encoder.decode([t]) for t in tokens_nandu], " <- la ñ quedó partida en dos tokens: cortar ahí produce '�'")

# Castellano vs. inglés: la misma oración
es = "La ventana de contexto limita cuánto texto puede procesar un modelo en cada llamada. Por eso conviene dividir los documentos en fragmentos pequeños y recuperar solo los más relevantes para cada pregunta."
en = "The context window limits how much text a model can process in each call. That is why it is convenient to split documents into small fragments and retrieve only the most relevant ones for each question."
for idioma, t in [("castellano", es), ("inglés", en)]:
    print(f"{idioma:10}: {contar_tokens(t)} tokens, {len(t) / contar_tokens(t):.2f} caracteres por token")


def chunking_por_tokens(texto: str, max_tokens: int = 150, solapamiento_tokens: int = 20) -> list[str]:
    """Arma chunks con PALABRAS COMPLETAS y CUENTA tokens para decidir dónde cortar:
    nunca parte una palabra (ni un carácter) al medio."""
    palabras = texto.split(" ")
    chunks, actual = [], []
    for palabra in palabras:
        if actual and contar_tokens(" ".join(actual + [palabra])) > max_tokens:
            chunks.append(" ".join(actual))
            arrastre = []                                   # solapamiento: últimas palabras, hasta ~solapamiento_tokens
            for w in reversed(actual):
                if contar_tokens(" ".join([w] + arrastre)) > solapamiento_tokens:
                    break
                arrastre.insert(0, w)
            actual = arrastre + [palabra]
        else:
            actual.append(palabra)
    if actual:
        chunks.append(" ".join(actual))
    return chunks


# Comparación directa sobre el mismo documento
print("\nCaracteres en el documento:", len(documento_largo))
print("Tokens en el documento:    ", contar_tokens(documento_largo), f"-> {len(documento_largo) / contar_tokens(documento_largo):.2f} caracteres por token (castellano con muchos términos técnicos en inglés)")

chunks_por_caracteres = chunking_por_caracteres(documento_largo, tamano=300, solapamiento=50)
chunks_por_tokens = chunking_por_tokens(documento_largo, max_tokens=75, solapamiento_tokens=15)

print(f"\nchunking_por_caracteres(300 chars): {len(chunks_por_caracteres)} chunks, tokens por chunk: {[contar_tokens(c) for c in chunks_por_caracteres]}")
print(f"chunking_por_tokens(75 tokens):     {len(chunks_por_tokens)} chunks, tokens por chunk: {[contar_tokens(c) for c in chunks_por_tokens]}")

Notá que `chunking_por_caracteres` da un número de tokens **variable** por chunk (a veces bastante por encima o por debajo de lo esperado), mientras que `chunking_por_tokens` los mantiene acotados **en tokens de `cl100k_base`** — que es lo que querés si tu presupuesto de contexto (Clase 6) está en tokens, como siempre lo está. Como el tokenizador de tu proveedor puede contar distinto, dejá margen.

**🏠 Practicá**: para el corpus real del TP (MDN, con snippets de código), medí la relación caracteres/token real (`len(texto) / contar_tokens(texto)`) y compará contra el ~4 de la heurística para inglés — los bloques de código deberían bajar bastante esa relación.

## 11. Chunking semántico — implementado

Más allá de tamaño fijo (sección 6) o por tokens (sección 10), el **chunking semántico** corta donde el *significado* cambia: se embebe cada oración, se mide la **distancia** entre cada oración y la siguiente, y se corta donde esa distancia es **inusualmente alta**.

¿"Inusualmente alta" respecto de qué? Un umbral **absoluto** ("cortar si la similitud < 0.75") no sirve: por la anisotropía (sección 2), con Cohere las oraciones consecutivas de un mismo tema dan similitudes de ~0.2 a 0.5, así que 0.75 cortaría **en todas**. La práctica estándar (Kamradt, *5 Levels of Text Splitting*) es un umbral **relativo**: cortar donde la distancia supera un **percentil** de las distancias del propio documento. Además ponemos un tope de tamaño.

Es más costoso que los métodos anteriores: hay que embeber **cada oración** en la ingesta.

In [ ]:
import re

def dividir_en_oraciones(texto: str) -> list[str]:
    # Separador simple por punto seguido de espacio y mayúscula -- no es perfecto (no maneja abreviaturas),
    # pero alcanza para la demo. En producción conviene una librería de segmentación de oraciones real.
    oraciones = re.split(r"(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÑ¿¡])", texto.strip())
    return [o.strip() for o in oraciones if o.strip()]


def chunking_semantico(texto: str, percentil: float = 80, max_chars: int = 1500, mostrar: bool = False) -> list[str]:
    oraciones = dividir_en_oraciones(texto)
    if len(oraciones) <= 2:
        return [" ".join(oraciones)] if oraciones else []

    vectores = []
    for i in range(0, len(oraciones), 96):                # Cohere en Bedrock acepta hasta 96 textos por llamada
        vectores += embed_bedrock(oraciones[i:i + 96], input_type="search_document")

    distancias = [1 - similitud_coseno(vectores[i - 1], vectores[i]) for i in range(1, len(oraciones))]
    umbral = float(np.percentile(distancias, percentil))   # umbral RELATIVO al documento, no un número fijo

    chunks, actual = [], [oraciones[0]]
    for i in range(1, len(oraciones)):
        corte_semantico = distancias[i - 1] > umbral
        demasiado_largo = len(" ".join(actual + [oraciones[i]])) > max_chars
        if mostrar:
            print(f"  dist={distancias[i-1]:.3f} {'✂ CORTE' if corte_semantico else '       '}  {oraciones[i][:60]}")
        if corte_semantico or demasiado_largo:
            chunks.append(" ".join(actual))
            actual = [oraciones[i]]
        else:
            actual.append(oraciones[i])
    chunks.append(" ".join(actual))
    return chunks


# Caso 1: dos temas del MISMO dominio (API de Bedrock vs. seguridad de LLMs)
corpus_mismo_dominio = corpus_ejemplo + """

La inyección de prompts ocurre cuando contenido no confiable manipula las instrucciones del modelo.
El principio de mínimo privilegio limita qué herramientas puede invocar un agente según el contexto.
Nunca hay que confiar ciegamente en contenido recuperado de fuentes externas sin validarlo."""

# Caso 2: dos temas REALMENTE distintos
corpus_dos_temas = corpus_ejemplo + """

El mate se prepara llenando el recipiente con yerba hasta dos tercios. Antes de cebar, conviene humedecer la yerba con agua tibia para que no se queme. El agua no debe hervir: la temperatura ideal ronda los 75 grados."""

for nombre, texto in [("mismo dominio", corpus_mismo_dominio), ("temas distintos", corpus_dos_temas)]:
    print(f"--- {nombre} ---")
    chunks_semanticos = chunking_semantico(limpiar_texto(texto), percentil=80, mostrar=True)
    for i, c in enumerate(chunks_semanticos):
        print(f"[chunk {i}] {c[:80]}...")
    print()

Lo que muestran los dos casos:

- Con **temas realmente distintos** (Bedrock vs. el mate), la distancia en la frontera es claramente la más alta y el corte cae justo ahí.
- Con dos temas del **mismo dominio** (API de Bedrock vs. seguridad de LLMs), para el modelo la diferencia es chica: las distancias entre oraciones son parecidas y el corte puede caer en otro lado. Con textos tan cortos la señal es débil.
- Con un umbral por percentil **siempre** se corta algo (el 20% de las distancias más altas): es un criterio relativo. Por eso conviene también un tope y, sobre todo, **medir** si el chunking semántico mejora la recuperación en tu corpus (Clase 8) antes de pagarlo.

**🏠 Practicá**: probá `percentil=50` y `percentil=95` sobre `corpus_dos_temas`. Con el percentil bajo, ¿cuántos chunks salen? ¿Y con uno alto? Es el mismo compromiso de tamaño de `chunking_por_caracteres` (chunks muy chicos pierden contexto, muy grandes mezclan temas), solo que ahora el corte lo decide la semántica.

## 12. Laboratorio: pipeline de ingesta documental

**🏠 Tarea de la semana — entregar/completar antes de la Clase 5.**

**Checkpoint del proyecto integrador (Clase 4)**: pipeline de ingesta del corpus (limpieza, chunking y metadata) listo para alimentar el índice vectorial de la Clase 5.

### Consigna

1. Tomá el corpus de referencia provisto por la cátedra (o un subconjunto representativo, para no esperar horas de embeddings si el corpus es grande).
2. Aplicá `limpiar_texto` (o tu propia versión extendida con la practicá de la sección 5) a cada documento.
3. Chunkeá cada documento con la estrategia que prefieras (caracteres, párrafos, tokens o semántico) — justificá en un comentario por qué elegiste ese tamaño/overlap para este corpus en particular.
4. Construí un `Chunk` por fragmento, con `documento_id`, `posicion`, y al menos `fuente` (de la practicá de la sección 7).
5. Embebé todos los chunks con la función de tu proveedor **en lotes** (Cohere en Bedrock acepta hasta 96 textos por llamada) y con el tipo de entrada de *documento*, y guardá los vectores junto con su `chunk_id` en una estructura simple (una lista de dicts, o un `.json` / `.pkl` — en la Clase 5 esto se migra a un vector store real).

### Criterio de éxito

- Corriendo el pipeline sobre el corpus completo (o el subconjunto elegido) no tira ningún error.
- Cada chunk resultante tiene menos de ~1500 caracteres (si no, revisá el tamaño elegido).
- Podés tomar cualquier `chunk_id` y recuperar de qué documento y qué posición vino, sin ambigüedad.
- El código de ingesta no incluye ninguna lógica de "búsqueda" — eso es, a propósito, la Clase 5.

In [ ]:
def pipeline_ingesta(documentos: dict[str, str], tamano=300, solapamiento=50, embed_fn=embed_bedrock) -> list[dict]:
    """documentos: {documento_id: texto_crudo}. Devuelve una lista de {chunk_id, texto, vector, documento_id, posicion}."""
    # TODO: para cada documento, limpiar -> chunkear -> construir Chunk -> embeber (en batch, no uno por uno)
    # TODO: devolver la lista combinando los datos del Chunk con su vector correspondiente
    raise NotImplementedError


# Prueba (va a fallar hasta que completes el TODO):
# resultado = pipeline_ingesta({"doc-converse-api": corpus_ejemplo})
# print(len(resultado), "chunks ingeridos")

## Cierre

- Hoy construimos la mitad "offline" de un sistema RAG: limpieza, chunking con criterio, metadata trazable y embeddings — sin tocar todavía la parte de búsqueda.
- Próxima clase: vector stores — ahí vive el índice que el pipeline de hoy alimenta, y ahí empieza la mitad "online" (la consulta).
- Antes de la próxima clase: dejar el pipeline de la sección 12 funcionando sobre el corpus del TP.